### Word2Vec
word2vec is not a singular algorithm, rather, it is a family of model architectures and optimizations that can be used to learn word embeddings from large datasets. Embeddings learned through word2vec have proven to be successful on a variety of downstream natural language processing tasks.

In [2]:
import io
import re
import string
import tqdm

import numpy as np

import tensorflow as tf
from tensorflow.keras import layers

In [3]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

In [4]:
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [25]:
# Split sentence to tokens like splitting by word
sentence = "When training word2vec, you want your model to learn which words appear together positive pairs, but also which words don’t negative pairs.Negative sampling is a trick to efficiently teach the model what isn’t context, without having to compute over the entire vocabulary."
tokens = list(set(sentence.lower().split()))
print(len(tokens))
print(tokens)

36
['pairs.negative', 'over', 'which', 'the', 'to', 'without', 'your', 'isn’t', 'entire', 'is', 'model', 'having', 'word2vec,', 'vocabulary.', 'training', 'words', 'don’t', 'learn', 'but', 'positive', 'you', 'trick', 'also', 'together', 'want', 'teach', 'pairs,', 'when', 'context,', 'a', 'what', 'appear', 'sampling', 'compute', 'negative', 'efficiently']


In [26]:
# Create vocab with given tokens
vocab, index = {}, 1  # start indexing from 1
vocab['<pad>'] = 0  # add a padding token
for token in tokens:
  if token not in vocab:
    vocab[token] = index
    index += 1
vocab_size = len(vocab)
print(vocab)

{'<pad>': 0, 'pairs.negative': 1, 'over': 2, 'which': 3, 'the': 4, 'to': 5, 'without': 6, 'your': 7, 'isn’t': 8, 'entire': 9, 'is': 10, 'model': 11, 'having': 12, 'word2vec,': 13, 'vocabulary.': 14, 'training': 15, 'words': 16, 'don’t': 17, 'learn': 18, 'but': 19, 'positive': 20, 'you': 21, 'trick': 22, 'also': 23, 'together': 24, 'want': 25, 'teach': 26, 'pairs,': 27, 'when': 28, 'context,': 29, 'a': 30, 'what': 31, 'appear': 32, 'sampling': 33, 'compute': 34, 'negative': 35, 'efficiently': 36}


In [27]:
inverse_vocab = {index: token for token, index in vocab.items()}
print(inverse_vocab)

{0: '<pad>', 1: 'pairs.negative', 2: 'over', 3: 'which', 4: 'the', 5: 'to', 6: 'without', 7: 'your', 8: 'isn’t', 9: 'entire', 10: 'is', 11: 'model', 12: 'having', 13: 'word2vec,', 14: 'vocabulary.', 15: 'training', 16: 'words', 17: 'don’t', 18: 'learn', 19: 'but', 20: 'positive', 21: 'you', 22: 'trick', 23: 'also', 24: 'together', 25: 'want', 26: 'teach', 27: 'pairs,', 28: 'when', 29: 'context,', 30: 'a', 31: 'what', 32: 'appear', 33: 'sampling', 34: 'compute', 35: 'negative', 36: 'efficiently'}


In [28]:
# vectorize sentence
example_sequence = [vocab[word] for word in tokens]
print(example_sequence)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]


In [29]:
# Genarate Skip-Grams from one sentence
window_size = 2
positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
    example_sequence,
    vocabulary_size=vocab_size,
    window_size=window_size,
    negative_samples=0,
    seed=1
)

print(len(positive_skip_grams))


138


In [30]:
# print positive skip grams
for target, context in positive_skip_grams[:5]:
    print(f'''({target}, {context}):({inverse_vocab[target]}, {inverse_vocab[context]})''')

(18, 20):(learn, positive)
(14, 16):(vocabulary., words)
(25, 27):(want, pairs,)
(34, 36):(compute, efficiently)
(31, 29):(what, context,)


### Negative sampling for skip-gram

In [31]:
# Get target and context words for one positive skip-gram.
target_word, context_word = positive_skip_grams[0]

# Set the number of negative samples per positive context.
num_ns = 4

context_class = tf.reshape(tf.constant(context_word, dtype="int64"), (1, 1))
negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
    true_classes=context_class,  # class that should be sampled as 'positive'
    num_true=1,  # each positive skip-gram has 1 positive context class
    num_sampled=num_ns,  # number of negative context words to sample
    unique=True,  # all the negative samples should be unique
    range_max=vocab_size,  # pick index of the samples from [0, vocab_size]
    seed=SEED,  # seed for reproducibility
    name="negative_sampling"  # name of this operation
)
print(negative_sampling_candidates)
print([inverse_vocab[index.numpy()] for index in negative_sampling_candidates])

tf.Tensor([ 8  2 17  4], shape=(4,), dtype=int64)
['isn’t', 'over', 'don’t', 'the']


### Construct one training example

In [32]:

# Reduce a dimension so you can use concatenation (in the next step).
squeezed_context_class = tf.squeeze(context_class, 1)

# Concatenate a positive context word with negative sampled words.
context = tf.concat([squeezed_context_class, negative_sampling_candidates], 0)

# Label the first context word as `1` (positive) followed by `num_ns` `0`s (negative).
label = tf.constant([1] + [0]*num_ns, dtype="int64")
target = target_word

In [33]:
# Check out the context and the corresponding labels for the target word from the skip-gram example above:
print(f"target_index    : {target}")
print(f"target_word     : {inverse_vocab[target_word]}")
print(f"context_indices : {context}")
print(f"context_words   : {[inverse_vocab[c.numpy()] for c in context]}")
print(f"label           : {label}")

target_index    : 18
target_word     : learn
context_indices : [20  8  2 17  4]
context_words   : ['positive', 'isn’t', 'over', 'don’t', 'the']
label           : [1 0 0 0 0]


In [34]:
print("target  :", target)
print("context :", context)
print("label   :", label)

target  : 18
context : tf.Tensor([20  8  2 17  4], shape=(5,), dtype=int64)
label   : tf.Tensor([1 0 0 0 0], shape=(5,), dtype=int64)


In [38]:
## create a vector using embedding 
embedding_dim = 64
embedding_layer = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

word = "over"
word_index = vocab[word]

vector = embedding_layer(tf.constant([word_index]))

vector_np = vector.numpy()[0]
print(vector_np)

[ 9.9186189e-03 -2.3155464e-02  4.5496114e-03 -3.0595470e-02
  3.5160508e-02  2.1460857e-02 -4.8375227e-02 -1.4937066e-02
  1.7001476e-02  4.0970210e-02  3.4702960e-02 -1.8055033e-02
 -2.4760162e-02  5.4500103e-03 -1.0016967e-02 -4.8328415e-03
 -3.9028358e-02  3.6014918e-02 -3.5590481e-02 -6.2420145e-03
  3.8477886e-02 -7.5436607e-03 -7.2062016e-05 -2.7882183e-02
  5.2564852e-03 -1.3217162e-02 -1.0800980e-02  3.4956064e-02
  4.9817946e-02  3.7517395e-02  4.6017442e-02  8.7698810e-03
  3.5853516e-02  1.8278610e-02  2.4307076e-02 -2.2357590e-03
  1.2975659e-02 -1.7849743e-02 -1.8244971e-02 -3.8401045e-02
  5.0059669e-03  1.1577450e-02  3.2478247e-02  6.4615123e-03
  5.8186427e-03  2.8322015e-02 -2.3276091e-02  1.4054965e-02
  1.7783556e-02  2.2466291e-02  3.4652520e-02  2.7095769e-02
  8.9126118e-03  2.0491373e-02  1.1869587e-02 -3.2365240e-02
 -2.1597398e-02 -1.3490804e-03  2.0653713e-02 -5.0126798e-03
  2.1303300e-02 -3.5045631e-03  3.7462357e-02 -2.3428930e-02]
